In [1]:
from utilsforecast.preprocessing import fill_gaps
from tinyshift.stats import remove_leading_zeros, is_obsolete
from tinyshift.plot import stationarity_analysis, pami, residual_analysis, seasonal_decompose
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from statsmodels.tsa.seasonal import MSTL
from statsforecast.models import SeasonalNaive, AutoETS
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from mlforecast import MLForecast
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, bias, cfe
from tinyshift.modelling import DMSTLWrapper, fourier_seasonality
from mlforecast.utils import PredictionIntervals
from tinyshift.series import (
    wape,
    pbias,
    score,
    forecast_instability,
    detect_seasonal_periods,
    extract_mstl_components,
    fva_rmae,
    mach,
    permutation_auto_mutual_information,
)
from dmstlv2 import DMSTLWrapperV2
from utilsforecast.preprocessing import fill_gaps

/home/heylucasleao/forecasting/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
df = df.groupby("unique_id")[df.columns].apply(remove_leading_zeros).reset_index(drop=True)
df = fourier_seasonality(df, "ds", seasonality=["monthly"])
days_obsoletes=180
obsolete_series = df.groupby("unique_id")[df.columns].apply(is_obsolete, days_obsoletes)
obsolote_ids = obsolete_series[obsolete_series].index.tolist()
assert len(obsolote_ids) == 0, f"Obsolete series found: {obsolote_ids}"

In [3]:
df.isnull().sum()

unique_id      0
ds             0
y              0
monthly_sin    0
monthly_cos    0
dtype: int64

In [4]:
from pathlib import Path


dataset_dir = Path(kagglehub.dataset_download("mashlyn/online-retail-ii-uci"))
retail_files = list(dataset_dir.rglob("*.csv"))
raw_retail = pd.read_csv(retail_files[0], encoding="latin1")
price_col = "Price" if "Price" in raw_retail.columns else "UnitPrice"
retail = raw_retail[["StockCode", "InvoiceDate", "Quantity", price_col]].copy()
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"], errors="coerce")
retail["StockCode"] = retail["StockCode"].astype(str)
retail = retail.loc[
    retail["InvoiceDate"].notna()
    & (retail["Quantity"] > 0)
    & (retail[price_col] > 0)
].copy()
retail["ds"] = retail["InvoiceDate"].dt.floor("D")
retail = retail.rename(columns={"StockCode": "unique_id"})
benchmark_skus = (
    retail.groupby("unique_id")["Quantity"]
    .sum()
    .nlargest(12)
    .index
    .tolist()
)
dates = pd.date_range(retail["ds"].min(), retail["ds"].max(), freq="D")
benchmark_values = (
    retail.groupby(["ds", "unique_id"])["Quantity"]
    .sum()
    .rename("y")
    .reset_index()
    
)
retail = pd.merge(benchmark_values, retail, on=["ds", "unique_id"], how="left")
benchmark_panel = benchmark_values[benchmark_values["unique_id"].isin(benchmark_skus)].copy()
retail = retail.loc[retail["unique_id"].isin(benchmark_skus), ["ds", "unique_id", "Price"]]

In [5]:
benchmark_panel = fill_gaps(
    benchmark_panel,
    freq="D",
    end="per_serie",
    id_col="unique_id",
    time_col="ds",
)
benchmark_panel["y"] = benchmark_panel["y"].fillna(0.0)
benchmark_cutoff = benchmark_panel["ds"].quantile(0.80)
benchmark_train = benchmark_panel[benchmark_panel["ds"] <= benchmark_cutoff].copy()
benchmark_test = benchmark_panel[benchmark_panel["ds"] > benchmark_cutoff].copy()
benchmark_horizon = 28

In [6]:
from utilsforecast.feature_engineering import fourier


In [ ]:
def even_day(dates):
    """Day of month is even"""
    return dates.day % 2 == 0

def month_start_or_end(dates):
    """Date is month start or month end"""
    return dates.is_month_start | dates.is_month_end

def is_monday(dates):
    """Date is monday"""
    return dates.dayofweek == 0

def residual_model_callable(nlags, freq):
    return MLForecast(
        models=[RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1)],
        freq=freq,
        lags=nlags,
        date_features=["dayofweek", "month", "quarter", "year", "dayofyear", even_day, month_start_or_end, is_monday]
    )


dmstl = DMSTLWrapperV2(
    residual_model_callable=residual_model_callable,
    freq="D",
    seasonal_detection_params={"top_k": 2, "fallback": [7]},
    nlags="auto",
    pami_params={"max_tau": 365, "m": 3, "delay": 1, "return_mode": "short_term", "fallback": 1},
)
train_df, future_x_df = fourier(
    df=benchmark_train, 
    freq='D', 
    season_length=28, 
    k=3, 
    h=28
)
dmstl.fit(train_df, target_col="y", static_features=[])

In [25]:
dmstl.mf_resid_

MLForecast(models=[RandomForestRegressor], freq=D, lag_features=['lag1', 'lag7', 'lag14', 'lag28'], date_features=['dayofweek', 'month', 'quarter', 'year', 'dayofyear', <function even_day at 0x71874d3fb010>, <function month_start_or_end at 0x71874d3facb0>, <function is_monday at 0x71874d3fba30>], num_threads=1)

In [26]:
result = dmstl.predict(h=28, X_df=future_x_df)
result = benchmark_test.merge(
    result[["unique_id", "ds", "RandomForestRegressor"]],
    on=["unique_id", "ds"],
)
result["RandomForestRegressor"] = result["RandomForestRegressor"].clip(lower=0)

In [27]:
evaluate(result, 
         metrics=[forecast_instability], 
         models=["RandomForestRegressor"], 
         id_col="unique_id", 
         time_col="ds", 
         target_col="y").sort_values(by="RandomForestRegressor", ascending=False)

,unique_id,metric,RandomForestRegressor
0,17003,forecast_instability,148.035049
4,22492,forecast_instability,137.716448
6,84077,forecast_instability,135.629861
10,85123A,forecast_instability,120.747235
3,22197,forecast_instability,115.293022
1,21212,forecast_instability,110.326606
7,84879,forecast_instability,101.616605
8,84991,forecast_instability,97.738937
9,85099B,forecast_instability,77.640091
2,21977,forecast_instability,66.507969
